# Legacy Flow experiment — superseded

The user reported buzzing after training this custom Flow student. Its quality is unvalidated.
Use [the pretrained TITL 8M adaptation notebook](https://colab.research.google.com/github/Huyvux12/Ancaptritue/blob/main/notebooks/02_MaiChi_Train_RTX6000.ipynb) for the new experiment. This file is retained for reproducibility only.

# 02 — RTX 6000 training from the saved Drive corpus

Finish [01 — T4 data](https://colab.research.google.com/github/Huyvux12/Ancaptritue/blob/main/notebooks/01_MaiChi_Data_T4.ipynb) first. This notebook restores COMPLETE data to SSD, trains vocoder then DiT/OT-CFM acoustic model, exports and evaluates. It does not generate training audio or load ZeroTTS teacher weights.

Strict complete-student budget: 9,518,564 parameters at vocabulary 256 including waveform vocoder. Initialization/export/reload reject >=10M. This is an experimental from-scratch model, not a proven best-quality checkpoint.

## 1. Configure the RTX runtime

Same mode/Drive folder as notebook 01. External runtimes may require an already mounted/synced folder and `USE_DRIVE=False`. Train from SSD; Drive is used for transfers/backups. Check free local and Drive storage.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
os.environ["ORT_DISABLE_TELEMETRY"] = "1"  # Before ONNX Runtime initialization
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
MODE = "pilot"  # SAME mode in both notebooks; "full" for the main experiment
USE_DRIVE = True
DRIVE_PARENT = "/content/drive/MyDrive/MaiChi_Distillation_v03"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
# External runtime: USE_DRIVE=False, DRIVE_PARENT=an already mounted/synced Drive folder
DATA_BUNDLE_DIR = Path(DRIVE_PARENT) / "datasets" / MODE
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(WORK)
assert MODE in {"pilot", "full"}
ROOT = WORK / "maichi_train_v03" / MODE
ROOT.mkdir(parents=True, exist_ok=True)
RUN_LABEL = "flow10m_default"  # Changed experiment: new label AND local ROOT
TRAIN_BACKUP_DIR = Path(DRIVE_PARENT) / "training" / MODE / RUN_LABEL
TRAIN_DEVICE = "cuda"
TEACHER_DEVICE = "cpu"  # Dependency selection only; teacher is not invoked here
CPU_THREADS = 4
BATCH_SIZE = 32
TRAIN_WORKERS = 4  # Use 0 if multiprocessing is unavailable
BACKUP_SECONDS = 600
RUN_ABLATION = False
RUN_BENCHMARK = MODE == "full"
RUN_UTMOS = False
VOCODER_STEPS = 2000 if MODE == "pilot" else 100000
FLOW_STEPS = 4000 if MODE == "pilot" else 200000
FLOW_NFE = [4, 8, 16]
assert (DATA_BUNDLE_DIR / "bundle.json").exists(), "Finish notebook 01 first"
assert json.loads((DATA_BUNDLE_DIR / "bundle.json").read_text())["complete"], "Data bundle is partial"
print({"bundle": str(DATA_BUNDLE_DIR), "local_root": str(ROOT), "backup": str(TRAIN_BACKUP_DIR)})
print("Local free GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## 2. Clone the exact producer commit

The saved bundle chooses the source commit automatically. Do not mix source/configuration changes into an existing experiment.

In [ ]:
REPO_URL = "https://github.com/Huyvux12/Ancaptritue.git"
p = DATA_BUNDLE_DIR / "bundle.json"
saved_manifest = json.loads(p.read_text()) if p.exists() else {}
REPO_REF = saved_manifest.get("source_commit") or "main"
REPO_DIR = WORK / "Ancaptritue"
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), "Different repository"
    assert not subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip(), "Preserve source edits before updating"
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_SHA = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
import hashlib
SOURCE_FILES = ["maichi_pipeline.py", "maichi_flow.py", "maichi_teacher_batch.py", "maichi_handoff.py", "maichi_train_runner.py"]
SOURCE_HASHES = {name: hashlib.sha256((REPO_DIR / name).read_bytes()).hexdigest() for name in SOURCE_FILES}
REVISION = {"repository": REPO_URL, "commit": REPO_SHA, "files": SOURCE_HASHES}
print("Pinned source:", REPO_SHA)


## 3. Install and check CUDA

Keep runtime CUDA PyTorch. ONNX CPU is an evaluation dependency; no teacher model is loaded. Actual CUDA kernels, GPU name and VRAM are checked. BF16 is selected when supported.

In [ ]:
def pip_install(*packages, extra=None):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(packages)
    if extra:
        cmd += list(extra)
    subprocess.run(cmd, check=True)

pip_install("setuptools==80.9.0", "wheel", "cython==3.0.12", "numpy==2.2.6")
pip_install(extra=["-r", str(REPO_DIR / "requirements-colab.txt")])
os.environ["CC"], os.environ["CXX"] = "gcc", "g++"
pip_install("pyworld==0.3.5", extra=["--no-build-isolation"])
pip_install("zerotts==0.1.5", extra=["--no-deps"])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"], check=True)
if TEACHER_DEVICE == "cuda":
    import torch
    cuda_major = int(torch.version.cuda.split(".")[0]) if torch.version.cuda else 0
    assert cuda_major in {12, 13}, "Use a CUDA 12/13 PyTorch runtime for teacher GPU inference"
    ORT_GPU_PIN = "1.30.0" if cuda_major == 13 else "1.23.2"
    pip_install(f"onnxruntime-gpu=={ORT_GPU_PIN}")
    print("ONNX GPU pin for CUDA", cuda_major, ":", ORT_GPU_PIN)
else:
    pip_install("onnxruntime==1.23.2")


In [ ]:
import torch, numpy as np, onnxruntime as ort
ort.disable_telemetry_events()
assert torch.cuda.is_available(), "No GPU visible to this kernel. Connect the intended GPU runtime first."
# Real CUDA operations catch missing kernel/architecture support early.
x = torch.randn(512, 512, device="cuda")
_ = x @ x
_ = torch.nn.Conv1d(16, 16, 3).cuda()(torch.randn(2, 16, 128, device="cuda"))
torch.cuda.synchronize()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Torch/CUDA:", torch.__version__, torch.version.cuda)
print("ONNX providers:", ort.get_available_providers())
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()  # Reuse compatible CUDA/cuDNN libraries loaded by PyTorch.
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
(ROOT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))


## 4. Import data, verify budget, restore checkpoints

Checksums are verified before extraction. Paths are rebound to this SSD. Incomplete/corrupt bundles stop here; a different bundle cannot replace an existing training run. Normalization statistics use TRAIN only.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
for name in ["maichi_pipeline", "maichi_flow", "maichi_teacher_batch", "maichi_handoff", "maichi_train_runner"]:
    sys.modules.pop(name, None)
import maichi_pipeline as mp
import maichi_handoff as handoff

def launch(filename, stage=None, **kwargs):
    cmd = [sys.executable, "-u", str(REPO_DIR / filename)]
    if stage is not None: cmd.append(stage)
    cmd += ["--root", str(ROOT)]
    for key, value in kwargs.items():
        flag = "--" + key.replace("_", "-")
        if isinstance(value, bool):
            if value: cmd.append(flag)
        elif value is not None: cmd += [flag, str(value)]
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

def data_run(stage, **kwargs): return launch("maichi_pipeline.py", stage, **kwargs)
import maichi_flow as mf
from maichi_train_runner import run_training
bundle = handoff.restore(DATA_BUNDLE_DIR, ROOT, require_complete=True)
assert bundle["source_hashes"] == SOURCE_HASHES, "Data/training source mismatch"
mp.atomic_json(ROOT / "training_source_revision.json", REVISION)
subprocess.run([sys.executable, str(REPO_DIR / "test_maichi_flow.py")], check=True)
vocab = json.loads((ROOT / "vocab.json").read_text())
model = mf.FlowStudent(len(vocab)); budget = mf.assert_budget(model)
mp.atomic_json(ROOT / "student_parameter_budget.json", budget)
print(budget); del model
handoff.restore_training(TRAIN_BACKUP_DIR, ROOT)
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"


## 5. Train waveform vocoder

Training reads SSD. Drive backups run every 10 minutes and after each stage, retaining objects referenced by the latest TWO successful snapshots. Progress since the last saved/copied checkpoint can be lost on VM termination. Best/last checkpoints are chosen/restored by validation, not public test scores.

In [ ]:
run_training(ROOT, TRAIN_BACKUP_DIR, backup_seconds=BACKUP_SECONDS,
             phase="vocoder", run="vocoder", steps=VOCODER_STEPS, batch_size=BATCH_SIZE,
             workers=TRAIN_WORKERS, device=TRAIN_DEVICE, precision=PRECISION,
             save_every=500 if MODE == "pilot" else 1000, eval_every=500 if MODE == "pilot" else 1000,
             allow_small=MODE == "pilot")


## 6. Stage B — train the conditional-flow acoustic model

A compact text encoder predicts a mel prior and duration. MAS supplies monotonic training alignment; an early diagonal preference decays away over 10,000 steps. A 10-block, width-192 DiT learns the OT-CFM velocity in normalized mel space. At inference, text predicts duration; an ODE sampler generates mel; the trained compact vocoder produces waveform.

This differs from the old VITS posterior/latent-flow/GAN speech model. The waveform decoder is still a convolutional vocoder, a common component in modern mel-based TTS. The implementation is a small experimental adaptation, not an official F5/Matcha/ZipVoice checkpoint. More data/steps or different hyperparameters may be necessary for acceptable quality.

In [ ]:
EXPERIMENTS = [("flow10m", 0.2)]
if RUN_ABLATION: EXPERIMENTS.append(("flow10m_no_pitch", 0.0))
for name, pitch_weight in EXPERIMENTS:
    run_training(ROOT, TRAIN_BACKUP_DIR, backup_seconds=BACKUP_SECONDS,
                 phase="acoustic", run=name, vocoder_run="vocoder", steps=FLOW_STEPS,
                 batch_size=BATCH_SIZE, workers=TRAIN_WORKERS, device=TRAIN_DEVICE,
                 precision=PRECISION, pitch_weight=pitch_weight,
                 save_every=500 if MODE == "pilot" else 1000, eval_every=500 if MODE == "pilot" else 1000,
                 allow_small=MODE == "pilot")


## 7. Inspect loss curves and listen to exported students

`best.pt` is selected by validation loss, not test quality. `student_deploy.pt` contains exactly one acoustic/vocoder state, the vocabulary and normalization buffers. Export and reload both enforce the budget. No teacher duration, F0, audio or latent is used for free-running synthesis.

Pilot speech is not expected to sound natural. The 4/8/16-step samples compare inference effort at the same parameter count, not additional model sizes.

In [ ]:
from IPython.display import Audio, display
import pandas as pd
import matplotlib.pyplot as plt
for name in ["vocoder"] + [e[0] for e in EXPERIMENTS]:
    history = pd.read_json(ROOT / "flow_runs" / name / "train.jsonl", lines=True)
    columns = [c for c in ["loss", "flow", "prior", "duration", "pitch", "mel", "stft"] if c in history]
    history.plot(x="step", y=columns, subplots=True, figsize=(10, 2 * len(columns)), title=name)
    plt.tight_layout(); plt.show()
for name, _ in EXPERIMENTS:
    path = ROOT / "flow_runs" / name / "student_deploy.pt"
    model, vocabulary = mf.load_deployment(path, TRAIN_DEVICE)
    print(name, mf.assert_budget(model), "MiB:", round(path.stat().st_size / 2**20, 2))
    text = "Xin chào, hôm nay chúng ta cùng tìm hiểu trí tuệ nhân tạo."
    ids = mp.encode(text, vocabulary)[None].to(TRAIN_DEVICE)
    for nfe in FLOW_NFE:
        audio = model.speak(ids, torch.tensor([ids.shape[1]], device=TRAIN_DEVICE), steps=nfe)[0, 0].float().cpu().numpy()
        print("Flow steps:", nfe)
        display(Audio(audio, rate=mp.SR))
    del model
    torch.cuda.empty_cache()


## 8. Student benchmark with saved teacher WAVs

Only student inference runs here. All 137 teacher WAVs come from notebook 01. Quality comparison uses matching text; teacher T4 and student RTX RTF are different-hardware measurements, not a speedup estimate.

In [ ]:
BENCH_NAMES = [f"{name}_nfe{nfe}" for name, _ in EXPERIMENTS for nfe in FLOW_NFE]
if RUN_BENCHMARK:
    manifest = mp.read_jsonl(ROOT / "benchmark_manifest.jsonl")
    assert len(manifest) == 137
    assert all((ROOT / "benchmark_wavs/teacher_maichi" / r["output_wav"]).exists() for r in manifest), "Generate teacher benchmark in notebook 01 before creating this training run"
    data_run("benchmark")  # Evaluator/dataset only, no teacher weights
    for name, _ in EXPERIMENTS:
        for nfe in FLOW_NFE:
            launch("maichi_flow.py", "synthesize", run=name, nfe=nfe, device=TRAIN_DEVICE, threads=CPU_THREADS)
            mp.atomic_json(ROOT / "benchmark_wavs" / f"{name}_nfe{nfe}" / "hardware.json", {
                "gpu": torch.cuda.get_device_name(0), "device": TRAIN_DEVICE, "timing_protocol": "single request on training GPU"})


## 9. Official quality scorer and adapted Mai Chi similarity

The official scorer retains the two ASR transcripts, expanded references, per-sample WER and silence metrics. UTMOSv2 is optional and missing values remain missing. Its original reference-speaker similarity is invalid for fixed Mai Chi and is not included as Mai Chi similarity in the comparison table. A separate WavLM cosine compares student to teacher Mai Chi on the same text.

No score is invented for steps that have not been executed. Automatic metrics do not replace blind listening, particularly for Vietnamese lexical tone.

In [ ]:
if RUN_UTMOS and RUN_BENCHMARK:
    pip_install("git+https://github.com/sarulab-speech/UTMOSv2.git")
    (ROOT / "pip_freeze_with_utmos.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
if RUN_BENCHMARK:
    for name in ["teacher_maichi"] + BENCH_NAMES:
        data_run("score", name=name, device="cuda", with_utmos=RUN_UTMOS)
    for name in BENCH_NAMES:
        data_run("similarity", name=name, device="cuda")
    data_run("report")
    comparison = pd.read_csv(ROOT / "comparison.csv")
    comparison["gpu"] = [json.loads((ROOT / "benchmark_wavs" / name / "hardware.json").read_text()).get("gpu") for name in comparison["run"]]
    comparison["cross_hardware_rtf_not_comparable"] = True
    comparison.to_csv(ROOT / "comparison.csv", index=False)
    display(comparison)


## 10. Blinded listening worksheet

Use 30 unique texts per comparison, randomized A/B order, separate keys, and report the listener protocol. Ask about intelligibility, tone errors, naturalness and voice identity. No quality-equivalence claim is supported until actual results are collected.

In [ ]:
if RUN_BENCHMARK:
    import random
    rng = random.Random(2026)
    manifest = mp.read_jsonl(ROOT / "benchmark_manifest.jsonl")
    unique = list({mp.text_key(r["text"]): r for r in manifest}.values())
    chosen = rng.sample(unique, min(30, len(unique)))
    for name in BENCH_NAMES:
        folder = ROOT / "listening" / name
        folder.mkdir(parents=True, exist_ok=True)
        ratings, key = [], []
        for i, row in enumerate(chosen):
            a, b = ("teacher_maichi", name) if rng.random() < 0.5 else (name, "teacher_maichi")
            for label, run_name in [("A", a), ("B", b)]:
                shutil.copy2(ROOT / "benchmark_wavs" / run_name / row["output_wav"], folder / f"{i:03d}_{label}.wav")
            ratings.append({"pair": i, "text": row["text"], "A_naturalness_1_to_5": "", "B_naturalness_1_to_5": "",
                            "A_tone_errors": "", "B_tone_errors": "", "preferred": "", "notes": ""})
            key.append({"pair": i, "id": row["id"], "A": a, "B": b})
        pd.DataFrame(ratings).to_csv(folder / "ratings.csv", index=False)
        pd.DataFrame(key).to_csv(folder / "key_keep_separate.csv", index=False)
print("Results directory:", ROOT)


## 11. Final backup/results

Checkpoints/deployment are periodically backed up. This final archive stores evaluation/listening outputs separately, excluding imported training data and teacher weights. An interrupted evaluation can be rerun without retraining.

In [ ]:
handoff.sync_training(ROOT, TRAIN_BACKUP_DIR)
import tarfile
archive = ROOT / "evaluation_results.tar"
with tarfile.open(archive, "w") as tar:
    for folder in ["scores", "benchmark_wavs", "listening"]:
        if (ROOT / folder).exists(): tar.add(ROOT / folder, arcname=folder)
    if (ROOT / "comparison.csv").exists(): tar.add(ROOT / "comparison.csv", arcname="comparison.csv")
handoff.atomic_copy(archive, TRAIN_BACKUP_DIR / "evaluation_results.tar")
print("Backups/results:", TRAIN_BACKUP_DIR)


## Validation scope

CPU checks cover real teacher batching, portable shard transfer, corruption/incomplete-bundle rejection, checkpoint backup/resume and the strict <10M student. Actual T4/RTX performance and full student quality remain unverified. Both notebooks run CUDA checks on your hardware.

100k vocoder + 200k acoustic steps are starting budgets, not guaranteed convergence. Inspect validation/listening; do not tune on the public test set.